# PokouAI — Fine-tune Gemma 4 on Cocoa Diseases

QLoRA fine-tune of Gemma 4 (E2B primary / E4B premium) on cocoa pod images across 6 disease classes and 4 languages, exported as GGUF Q4_K_M for on-device inference.

- **Platform**: Kaggle T4 x2 (Unsloth uses one T4)
- **Stack**: Unsloth + PEFT + TRL
- **First-pass scope**: 5k train / 500 val / 1 epoch — ~30 min smoke test. Bump constants once the pipeline is verified.

## 1 — Environment setup

In [ ]:
!pip install -qU "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qU transformers accelerate peft trl datasets bitsandbytes
# Pillow ABI fix MUST be the last install — unsloth's deps would otherwise pull
# Pillow 12.2 back in, which mismatches Kaggle's compiled _imaging.so (11.3).
!pip install --force-reinstall --no-cache-dir --no-deps "pillow==11.3.0"
print('✓ deps installed')

In [ ]:
import os, json, time, torch
from pathlib import Path
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

print(f'CUDA: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {p.total_memory / 1e9:.1f} GB')

## 2 — Load Gemma 4 (E2B primary, E4B optional)

Direct HF load (`google/gemma-4-{variant}-it`) — falls back to Unsloth's pre-quantized 4-bit repo if the direct load errors.

In [ ]:
VARIANT = 'e2b'   # 'e2b' (primary, ~2B) or 'e4b' (premium, ~4B)
MAX_SEQ_LEN = 2048

DIRECT_REPO   = f'google/gemma-4-{VARIANT}-it'
FALLBACK_REPO = f'unsloth/gemma-4-{VARIANT}-it-bnb-4bit'

def load_model():
    try:
        print(f'trying direct load: {DIRECT_REPO}')
        return FastModel.from_pretrained(
            model_name=DIRECT_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )
    except Exception as e:
        print(f'direct load failed ({type(e).__name__}: {e}) — falling back to {FALLBACK_REPO}')
        return FastModel.from_pretrained(
            model_name=FALLBACK_REPO,
            max_seq_length=MAX_SEQ_LEN,
            load_in_4bit=True,
            dtype=None,
        )

model, tokenizer = load_model()
print(f'✓ loaded {VARIANT.upper()}: {type(model).__name__}')

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=True,
    finetune_language_layers=True,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

## 3 — Load dataset (subsampled)

The full dataset is ~190k examples. First pass uses 5k/500 to validate the pipeline in ~30 min — bump `TRAIN_N` / `NUM_EPOCHS` once a smoke run produces a sane GGUF.

In [ ]:
from datasets import load_dataset, Image as ImageFeature

# Adjust DATA_ROOT if your Kaggle dataset slug differs
DATA_ROOT   = Path('/kaggle/input/datasets/yaokouadio/pokou-ai-cocoa-training-data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
VAL_JSONL   = DATA_ROOT / 'val.jsonl'

print(f'loading train: {TRAIN_JSONL}')
train_ds = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
print(f'loading val:   {VAL_JSONL}')
val_ds = load_dataset('json', data_files=str(VAL_JSONL), split='train')
print(f'\nfull dataset:    train={len(train_ds):,}   val={len(val_ds):,}')

# Subsample for the smoke run
TRAIN_N = 5_000
VAL_N   = 500
train_ds = train_ds.shuffle(seed=42).select(range(min(TRAIN_N, len(train_ds))))
val_ds   = val_ds.shuffle(seed=42).select(range(min(VAL_N,   len(val_ds))))
print(f'subsampled:      train={len(train_ds):,}   val={len(val_ds):,}')

In [ ]:
# Build messages with embedded PIL — but lazily.
#
# Unsloth's vision collator expects the actual PIL inside content as
# {"type": "image", "image": <PIL>}. We can't achieve that via .map()
# without serialising N PIL objects into the disk cache (the bug that
# blew the 20 GB Kaggle quota last run).
#
# `set_transform` runs the conversion on every row access — no cache, no
# materialisation. PIL.Image.open is called only when the DataLoader
# fetches a batch, ~8 images per training step.
#
# `thumbnail((896, 896))` caps the long edge at 896 px before the vision
# encoder sees it. Without this, full-resolution augmented images can hang
# the encoder for many minutes per batch.

from PIL import Image as PILImage

MAX_IMG = 896

def to_chat_transform(batch):
    out_messages = []
    out_images   = []
    for original in batch['messages']:
        image_rel      = next(c['path'] for c in original[0]['content'] if c['type'] == 'image')
        user_text      = next(c['text'] for c in original[0]['content'] if c['type'] == 'text')
        assistant_text = original[1]['content'][0]['text']
        pil = PILImage.open(str(DATA_ROOT / image_rel)).convert('RGB')
        pil.thumbnail((MAX_IMG, MAX_IMG))
        out_messages.append([
            {'role': 'user', 'content': [
                {'type': 'image', 'image': pil},
                {'type': 'text',  'text':  user_text},
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': assistant_text},
            ]},
        ])
        out_images.append(pil)
    return {'messages': out_messages, 'image': out_images}

train_ds.set_transform(to_chat_transform)
val_ds.set_transform(to_chat_transform)

# Sanity check — accesses trigger the lazy transform
sample = train_ds[0]
print(f'columns:          {list(sample.keys())}')
print(f'image type/size:  {type(sample["image"]).__name__} {sample["image"].size}')
print(f'msg[0] role:      {sample["messages"][0]["role"]}')
print(f'msg[0] content[0]: {sample["messages"][0]["content"][0].keys()}')
print(f'\n✓ ready: train={len(train_ds):,}  val={len(val_ds):,}  (PIL loaded lazily, capped at {MAX_IMG}px)')

## 4 — Training config

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator

OUTPUT_DIR = f'/kaggle/working/cocoa_v1_{VARIANT}'

# T4 (compute capability 7.5) does NOT support bf16 — use fp16.
# logging_steps=5 keeps the cell verbose so you can see training is alive.
# save_total_limit=1 caps disk usage at one checkpoint (~1.5 GB for E2B).
# eval_strategy='no' skips per-step eval — we evaluate at the end manually.
# Vision-specific: UnslothVisionDataCollator + skip_prepare_dataset +
#                  remove_unused_columns=False (so 'image' survives).
# dataloader_num_workers=0: workers can't pickle set_transform across forks
#                          on Kaggle; single-process loading is slower per
#                          step but won't hang. Trade ~20% throughput for
#                          reliability.

PER_DEVICE_BATCH = 4 if VARIANT == 'e2b' else 2
GRAD_ACCUM       = 2 if VARIANT == 'e2b' else 4
NUM_EPOCHS       = 1

print('training plan:')
print(f'  variant         = {VARIANT}')
print(f'  per-device bsz  = {PER_DEVICE_BATCH}')
print(f'  grad accum      = {GRAD_ACCUM}')
print(f'  effective bsz   = {PER_DEVICE_BATCH * GRAD_ACCUM}')
print(f'  epochs          = {NUM_EPOCHS}')
print(f'  steps/epoch     ≈ {len(train_ds) // (PER_DEVICE_BATCH * GRAD_ACCUM)}')
print(f'  output_dir      = {OUTPUT_DIR}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=5,
        eval_strategy='no',
        save_strategy='epoch',
        save_total_limit=1,
        bf16=False,
        fp16=True,
        max_seq_length=MAX_SEQ_LEN,
        report_to='none',
        seed=42,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        dataset_kwargs={'skip_prepare_dataset': True},
    ),
)
print('\n✓ trainer initialised (vision collator, single-process loader)')

## 5 — Train

In [ ]:
torch.cuda.empty_cache()
print(f'GPU mem before train: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'starting training at {time.strftime("%H:%M:%S")}\n')

t0 = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - t0

print(f'\n✓ training finished in {elapsed / 60:.1f} min')
print(f'  final loss      = {trainer_stats.training_loss:.4f}')
print(f'  total steps     = {trainer_stats.global_step}')
print(f'  GPU mem peak    = {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

## 6 — Save LoRA adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import subprocess
size = subprocess.check_output(['du', '-sh', OUTPUT_DIR]).decode().split()[0]
print(f'✓ LoRA adapter saved to {OUTPUT_DIR} ({size})')

## 7 — Export to GGUF Q4_K_M

- E2B Q4_K_M: ~1.3–1.6 GB (target — fits 2–3 GB RAM Android)
- E4B Q4_K_M: ~2.5–3.0 GB

In [ ]:
GGUF_OUTPUT = f'/kaggle/working/cocoa_v1_{VARIANT}_gguf'

print(f'exporting GGUF Q4_K_M to {GGUF_OUTPUT} (this can take 5–10 min)...')
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method='q4_k_m')
print('✓ export done\n')

!ls -lh {GGUF_OUTPUT}/
size = subprocess.check_output(['du', '-sh', GGUF_OUTPUT]).decode().split()[0]
print(f'\n✓ total: {size}')

## 8 — Quick sanity check (single-image inference)

In [ ]:
from transformers import TextStreamer

sample = val_ds[0]
inputs = tokenizer.apply_chat_template(
    sample['conversations'][:1],
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
).to('cuda')

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, max_new_tokens=400, streamer=streamer, do_sample=False)

## 9 — Train the other variant

To produce both models for the submission, restart the kernel, set `VARIANT = 'e4b'` in cell 5, and re-run from §3. Outputs land in a separate folder (`cocoa_v1_e4b_gguf/`), so nothing is overwritten.